In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Tesla T4, 15360 MiB


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [ ]:
!pip -q install \
  "transformers==4.46.*" \
  "accelerate==1.1.*" \
  "bitsandbytes"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 115.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [5]:
import subprocess
import threading
import time

_util_samples = []
_sampler_thread = None
_sampler_running = False


def _sample_gpu():
    global _sampler_running

    while _sampler_running:
        try:
            result = subprocess.check_output(
                [
                    "nvidia-smi",
                    "--query-gpu=utilization.gpu",
                    "--format=csv,noheader,nounits",
                ],
                text=True,
            )

            value = float(result.strip().splitlines()[0])
            _util_samples.append(value)

        except Exception:
            pass

        time.sleep(0.2)


def start_sampler():
    global _util_samples
    global _sampler_thread
    global _sampler_running

    _util_samples = []
    _sampler_running = True

    _sampler_thread = threading.Thread(
        target=_sample_gpu,
        daemon=True,
    )

    _sampler_thread.start()


def stop_sampler():
    global _sampler_running
    global _sampler_thread

    _sampler_running = False

    if _sampler_thread is not None:
        _sampler_thread.join()


def read_util_mean():
    if not _util_samples:
        return 0.0

    return sum(_util_samples) / len(_util_samples)

In [6]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Tesla T4, 15360 MiB


In [7]:
import time
import gc
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)


MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL)

# مهم للـbatching
tok.pad_token = tok.eos_token
tok.padding_side = "left"


def load(dtype: str):

    if dtype == "fp16":
        return AutoModelForCausalLM.from_pretrained(
            MODEL,
            torch_dtype=torch.float16,
            device_map="cuda",
        )

    if dtype == "int8":
        qc = BitsAndBytesConfig(
            load_in_8bit=True
        )

        return AutoModelForCausalLM.from_pretrained(
            MODEL,
            quantization_config=qc,
            device_map="cuda",
        )

    raise ValueError(dtype)


def make_prompt(context_tokens: int) -> str:

    base = (
        "Summarise the following text "
        "in one sentence.\n"
    )

    filler = (
        "The data center runs many small "
        "inference requests all day. " * 400
    )

    ids = tok(
        base + filler
    )["input_ids"][:context_tokens]

    return tok.decode(ids)


def resident_vram_gb() -> float:

    torch.cuda.synchronize()

    return (
        torch.cuda.memory_reserved()
        / (1024 ** 3)
    )


def profile(
    model,
    dtype: str,
    context: int,
    new_tokens: int = 128,
    batch: int = 1,
):

    prompt = make_prompt(context)

    prompts = [prompt] * batch

    enc = tok(
        prompts,
        return_tensors="pt",
        padding=True,
    ).to("cuda")

    # warm-up
    _ = model.generate(
        **enc,
        max_new_tokens=8,
        do_sample=False,
    )

    vram = resident_vram_gb()

    start_sampler()

    t0 = time.time()

    out = model.generate(
        **enc,
        max_new_tokens=new_tokens,
        do_sample=False,
    )

    dt = time.time() - t0

    stop_sampler()

    gen_tokens = (
        out.shape[1]
        - enc["input_ids"].shape[1]
    ) * batch

    return {
        "dtype": dtype,
        "context": context,
        "vram_gb": round(vram, 3),
        "util_mean": round(
            read_util_mean(),
            1,
        ),
        "tokens_per_s": round(
            gen_tokens / dt,
            1,
        ),
    }


def free_vram():

    gc.collect()

    torch.cuda.empty_cache()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
rows = []

for dtype in ["fp16", "int8"]:

    print(f"\nLoading {dtype} model...")

    model = load(dtype)

    for context in [
        512,
        2048,
        4096,
    ]:

        print(
            f"Profiling dtype={dtype}, "
            f"context={context}"
        )

        row = profile(
            model,
            dtype,
            context,
        )

        print(row)

        rows.append(row)

    del model
    free_vram()

    print(f"Freed VRAM after {dtype}")


Loading fp16 model...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Profiling dtype=fp16, context=512


...

{'dtype': 'int8', 'context': 512, 'vram_gb': 1.805, 'util_mean': 26.5, 'tokens_per_s': 6.0}
Profiling dtype=int8, context=2048


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarni

{'dtype': 'int8', 'context': 2048, 'vram_gb': 2.035, 'util_mean': 28.7, 'tokens_per_s': 5.6}
Profiling dtype=int8, context=4096


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarni

{'dtype': 'int8', 'context': 4096, 'vram_gb': 2.309, 'util_mean': 32.6, 'tokens_per_s': 5.0}
Freed VRAM after int8


In [9]:
model = load("fp16")

b1 = profile(
    model,
    "fp16",
    512,
    new_tokens=128,
    batch=1,
)

b8 = profile(
    model,
    "fp16",
    512,
    new_tokens=128,
    batch=8,
)

del model
free_vram()

print("batch 1:", b1)
print("batch 8:", b8)

print(
    "tokens/s ratio:",
    round(
        b8["tokens_per_s"]
        / b1["tokens_per_s"],
        2,
    ),
)

print(
    "util delta:",
    round(
        b8["util_mean"]
        - b1["util_mean"],
        1,
    ),
)

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarni

batch 1: {'dtype': 'fp16', 'context': 512, 'vram_gb': 3.113, 'util_mean': 56.2, 'tokens_per_s': 29.7}
batch 8: {'dtype': 'fp16', 'context': 512, 'vram_gb': 3.527, 'util_mean': 81.1, 'tokens_per_s': 187.4}
tokens/s ratio: 6.31
util delta: 24.9


In [10]:
import json

with open(
    "batch_check.json",
    "w",
) as f:
    json.dump(
        {
            "batch1_tokens_per_s": b1["tokens_per_s"],
            "batch8_tokens_per_s": b8["tokens_per_s"],
        },
        f,
        indent=2,
    )

print("wrote batch_check.json")

wrote batch_check.json


In [11]:
with open(
    "profile.json",
    "w",
) as f:
    json.dump(
        rows,
        f,
        indent=2,
    )

print(
    "wrote",
    len(rows),
    "rows to profile.json",
)

wrote 6 rows to profile.json


In [12]:
!python verify_cell.py

rows: 6, dtypes: ['fp16', 'int8'], contexts: [512, 2048, 4096]
batch-1 tokens/s: 29.7, batch-8 tokens/s: 187.4
GREEN CHECK: PASS


In [13]:
from google.colab import files

for f_ in [
    "profile.json",
    "batch_check.json",
]:
    files.download(f_)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
!mkdir -p w3d1-gpu-profiling
!mv profile.json batch_check.json verify_cell.py colab_scaffold.py w3d1-gpu-profiling/

In [15]:
!git clone https://github.com/zyadalsmeri/zyadalsmeri-Zyad-Alsmeri-serving-stack.git

Cloning into 'zyadalsmeri-Zyad-Alsmeri-serving-stack'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 52 (delta 11), reused 48 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 25.06 KiB | 1.25 MiB/s, done.
Resolving deltas: 100% (11/11), done.


In [16]:
%cd /content/zyadalsmeri-Zyad-Alsmeri-serving-stack

/content/zyadalsmeri-Zyad-Alsmeri-serving-stack


In [17]:
!git switch -c w3d1-gpu-profiling

Switched to a new branch 'w3d1-gpu-profiling'


In [19]:
!mkdir -p w3d1-gpu-profiling

In [22]:
!cp /content/profile.json w3d1-gpu-profiling/
!cp /content/batch_check.json w3d1-gpu-profiling/
!cp /content/verify_cell.py w3d1-gpu-profiling/
!cp /content/colab_scaffold.py w3d1-gpu-profiling/

cp: cannot stat '/content/profile.json': No such file or directory
cp: cannot stat '/content/batch_check.json': No such file or directory
cp: cannot stat '/content/verify_cell.py': No such file or directory
cp: cannot stat '/content/colab_scaffold.py': No such file or directory


In [23]:
!ls -la w3d1-gpu-profiling

total 12
drwxr-xr-x 3 root root 4096 Aug 30 12:19 .
drwxr-xr-x 8 root root 4096 Aug 30 12:17 ..
drwxr-xr-x 2 root root 4096 Aug 30 11:52 w3d1-gpu-profiling


In [24]:
!find /content -maxdepth 4 -type f \( \
  -name "profile.json" -o \
  -name "batch_check.json" -o \
  -name "verify_cell.py" -o \
  -name "colab_scaffold.py" \
\)

/content/zyadalsmeri-Zyad-Alsmeri-serving-stack/w3d1-gpu-profiling/w3d1-gpu-profiling/profile.json
/content/zyadalsmeri-Zyad-Alsmeri-serving-stack/w3d1-gpu-profiling/w3d1-gpu-profiling/verify_cell.py
/content/zyadalsmeri-Zyad-Alsmeri-serving-stack/w3d1-gpu-profiling/w3d1-gpu-profiling/batch_check.json
/content/zyadalsmeri-Zyad-Alsmeri-serving-stack/w3d1-gpu-profiling/w3d1-gpu-profiling/colab_scaffold.py


In [25]:
%cd /content/zyadalsmeri-Zyad-Alsmeri-serving-stack

!mv w3d1-gpu-profiling/w3d1-gpu-profiling/* w3d1-gpu-profiling/
!rmdir w3d1-gpu-profiling/w3d1-gpu-profiling

!ls -la w3d1-gpu-profiling

/content/zyadalsmeri-Zyad-Alsmeri-serving-stack
total 40
drwxr-xr-x 2 root root  4096 Aug 30 12:22 .
drwxr-xr-x 8 root root  4096 Aug 30 12:17 ..
-rw-r--r-- 1 root root    65 Aug 30 11:18 batch_check.json
-rw-r--r-- 1 root root 15703 Aug 30 11:00 colab_scaffold.py
-rw-r--r-- 1 root root   723 Aug 30 11:19 profile.json
-rw-r--r-- 1 root root  4296 Aug 30 11:00 verify_cell.py


In [26]:
!git status --short

?? w3d1-gpu-profiling/


In [27]:
!git add w3d1-gpu-profiling
!git status

On branch w3d1-gpu-profiling
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   w3d1-gpu-profiling/batch_check.json
	new file:   w3d1-gpu-profiling/colab_scaffold.py
	new file:   w3d1-gpu-profiling/profile.json
	new file:   w3d1-gpu-profiling/verify_cell.py



In [28]:
!git config user.name "zyadalsmeri"
!git config user.email "zyadalsmerii@gmail.com"
!git commit -m "Complete W3D1 GPU profiling lab"

[w3d1-gpu-profiling d6d6fe8] Complete W3D1 GPU profiling lab
 4 files changed, 529 insertions(+)
 create mode 100644 w3d1-gpu-profiling/batch_check.json
 create mode 100644 w3d1-gpu-profiling/colab_scaffold.py
 create mode 100644 w3d1-gpu-profiling/profile.json
 create mode 100644 w3d1-gpu-profiling/verify_cell.py


In [29]:
!git push -u origin w3d1-gpu-profiling

fatal: could not read Username for 'https://github.com': No such device or address


In [30]:
from google.colab import files

files.download("profile.json")
files.download("batch_check.json")
files.download("verify_cell.py")
files.download("colab_scaffold.py")

FileNotFoundError: Cannot find file: profile.json